In [1]:
from rdflib import Graph, Namespace, RDF
import networkx as nx
from pyvis.network import Network

from matplotlib import cm
import random




In [ ]:

class NetworkGraph:
    def __init__(self):
        self.graph = nx.DiGraph()
        self.node_types = {}


class OntoVis:


    def __init__(self, ontologies_pathes = [r'D:\github_repos\POKIMON\visualization\test\testing.owl'], color_map_instance = {}):
        # Load ontology
        global_graph = Graph()
        for path in ontologies_pathes:
            global_graph.parse(path)  # or .ttl, .rdf
        self.global_graph = global_graph
        self.color_map_instance = color_map_instance


    def get_sparql_query(self, query):
        return self.global_graph.query(query)
    

    def build_network_graph(self, query  ):
        g = self.global_graph
        query_results = self.get_sparql_query(query)
        #build the graph
        netwrk_graph = NetworkGraph()

        for row in query_results:
            subj_uri = row.subject
            pred_uri = row.predicate  # now valid
            obj_uri = row.object      # note just one object variable now

            subj = OntoVis.short(str(subj_uri))
            pred = OntoVis.short(str(pred_uri))
            obj = OntoVis.short(str(obj_uri))

            # Get and save subject type if not already done
            if subj not in netwrk_graph.node_types:
                netwrk_graph.node_types[subj] = self.get_node_type(subj_uri)

            # Add edge and set object type
            netwrk_graph.graph.add_edge(subj, obj, label=pred)
            if obj not in netwrk_graph.node_types:
                netwrk_graph.node_types[obj] = self.get_node_type(obj_uri)

        return netwrk_graph

    
    def generate_html_from_network_graph(self, query, ouput_html_path = "full_ontology_graph1.html"):
        
        network_graph = self.build_network_graph(query)

        # Visualize
        net = Network(height="700px", width="100%", directed=True)
        for node in network_graph.graph.nodes():
            node_type = network_graph.node_types.get(node, "Unknown")
            color = self.get_color_for_class(node_type)
            net.add_node(
                node,
                label=node,
                shape="dot",
                size=35,
                color=color,
                font={
                    "size": 12,
                    "color": "#000000" ,
                    "face": "arial",
                    "vadjust": -50  # center the label vertically
                }
            )

        for src, tgt, data in network_graph.graph.edges(data=True):
            net.add_edge(
                src,
                tgt,
                label=data.get("label", ""),
                arrows="to",
                font={"size": 10},
                length=250
            )

        net.write_html(ouput_html_path)    


    def get_node_type(self, uri ):
        g = self.global_graph
        types = set()
        for _, _, o in g.triples((uri, RDF.type, None)):
            _type = OntoVis.short(str(o))
            if _type != 'NamedIndividual':
                types.add(_type)
        if types:
            # Return first type or combine multiple if needed
            return list(types)[0]
        else:
            return "Unknown"
        
    def get_color_for_class(self, _cls):
        if _cls not in self.color_map_instance:
            self.color_map_instance[_cls] = f'#{random.randint(0, 0xFFFFFF):06x}'
        return self.color_map_instance[_cls]
    
    
    @staticmethod
    def short(uri):
        if uri is None:
            return "Unknown"
        return uri.split("#")[-1] if "#" in uri else uri.split("/")[-1]





In [3]:
# Colors for types
color_map = {
"Algorithm": "#0891f3",            # blue
"Planned_Process": "#188dd6",      # green
"Action_Specification": "#1780D6", # red
"Unknown": "#888"
}

pokimon_vis = OntoVis( color_map_instance = color_map)


In [4]:



POKI = Namespace("http://www.POKIMON#")

query = """

PREFIX poki: <http://www.POKIMON#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?subject ?predicate ?object
WHERE {
  ?subject rdf:type poki:Algorithm .

  {
    ?subject poki:is_About ?object1 .
    ?object1 rdf:type poki:Planned_Process .
  }
  FILTER EXISTS {
    ?subject poki:has_Part ?object2 .
    ?object2 rdf:type poki:Action_Specification .
  }

  {
    ?subject poki:is_About ?object .
    BIND(poki:is_About AS ?predicate)
  }
  UNION
  {
    ?subject poki:has_Part ?object .
    BIND(poki:has_Part AS ?predicate)
  }
}


"""




In [5]:
pokimon_vis.generate_html_from_network_graph(query)